# Introduction

> Distributions, installation, workspaces, the node graph, and the `ros2` command line.

- skip_showdoc: true
- skip_exec: true


## Distributions

ROS 2 ships a named distribution roughly once a year, each pinned to one Ubuntu
release. Picking one is the first decision, and it is mostly a question of which
Ubuntu is on the robot.

| Distribution | Ubuntu | Support |
|--------------|--------|---------|
| Humble Hawksbill | 22.04 | LTS, 5 years (to May 2027) |
| Jazzy Jalisco | 24.04 | LTS, 5 years (to May 2029) |
| Rolling Ridley | latest | rolling development, no stability promise |

Prefer an LTS distribution. Non-LTS releases get about 18 months, which is
shorter than most robot projects. Mixing distributions on one network works
in principle (the wire protocol is DDS) but message definitions drift between
releases, so treat it as something to avoid rather than rely on.

---


## Installing

The Debian packages are the normal path on Ubuntu. Add the ROS 2 apt repository,
then install the `desktop` metapackage (core plus RViz2, demos and tutorials) or
`ros-base` for a headless robot.

```bash
# Repository setup (once)
sudo apt install software-properties-common curl -y
sudo add-apt-repository universe
sudo curl -sSL https://raw.githubusercontent.com/ros/rosdistro/master/ros.key \
  -o /usr/share/keyrings/ros-archive-keyring.gpg
echo "deb [arch=$(dpkg --print-architecture) signed-by=/usr/share/keyrings/ros-archive-keyring.gpg] \
  http://packages.ros.org/ros2/ubuntu $(. /etc/os-release && echo $UBUNTU_CODENAME) main" \
  | sudo tee /etc/apt/sources.list.d/ros2.list > /dev/null

# Install (Jazzy on Ubuntu 24.04)
sudo apt update && sudo apt install ros-jazzy-desktop -y

# Build tooling
sudo apt install ros-dev-tools python3-colcon-common-extensions -y
```

Every shell needs the setup script sourced before `ros2` exists:

```bash
source /opt/ros/jazzy/setup.bash
```

Putting that line in `~/.bashrc` is convenient and is what most people do. It is
also the reason for a whole class of confusing bugs once there is more than one
distribution or more than one workspace on the machine, so know that it is there.

---


## Workspaces and Packages

Source code lives in a **workspace**: a directory with a `src/` folder holding
packages. `colcon build` compiles them into `build/`, `install/` and `log/`
siblings.

```bash
mkdir -p ~/ros2_ws/src
cd ~/ros2_ws
colcon build --symlink-install
source install/setup.bash
```

`--symlink-install` links Python files into `install/` instead of copying them,
so editing a Python node takes effect without rebuilding. C++ still needs a
rebuild.

Creating a package:

```bash
cd ~/ros2_ws/src
ros2 pkg create --build-type ament_python my_pkg      # Python
ros2 pkg create --build-type ament_cmake  my_cpp_pkg  # C++
```

A Python package is an ordinary Python package plus two manifests:

```
my_pkg/
├── package.xml        # name, version, dependencies (ROS metadata)
├── setup.py           # entry points - maps a name to a node's main()
├── resource/my_pkg    # marker file the ament index looks for
└── my_pkg/
    └── __init__.py
```

Dependencies declared in `package.xml` are installed with `rosdep`, which maps
ROS package names onto system packages:

```bash
rosdep install --from-paths src --ignore-src -r -y
```

Sourcing order matters: the distribution's `setup.bash` first, then the
workspace's `install/setup.bash`. The workspace **overlays** the distribution,
so a package built locally shadows the installed one of the same name.

---


## The Node Graph

A running system is a graph of nodes. There are four ways for nodes to exchange
information, and choosing the wrong one is the most common design mistake.

**Topics** are a named, typed, many-to-many stream. Publishers do not know who
is listening and get no acknowledgement. Use them for continuous data: sensor
readings, odometry, velocity commands.

**Services** are a blocking request/response call between one client and one
server. Use them for short operations with an answer: query a parameter, reset
a counter. Never for anything slow, since the client waits.

**Actions** are a long-running request with feedback and the ability to cancel.
Use them for goals that take seconds or minutes: navigate to a pose, follow a
trajectory. Internally an action is two services plus a feedback topic.

**Parameters** are per-node configuration values, settable at launch and
changeable at runtime. Each node owns its own.

Topics carry a **QoS profile** that decides the delivery guarantees. The two
settings that matter most:

- `reliability`: `RELIABLE` retries until delivery, `BEST_EFFORT` drops. Sensor
  data is usually best effort; commands are usually reliable.
- `durability`: `TRANSIENT_LOCAL` replays the last message to a subscriber that
  joins late, `VOLATILE` does not. Latched data like a map needs transient local.

A publisher and subscriber whose profiles are **incompatible** simply never
connect, with no error on either side. That silence is the single most common
"my topic is not working" cause.

---


## The CLI

`ros2` is the one entry point, with subcommands mirroring the concepts above.
These are the ones used constantly:

```bash
# What is running
ros2 node list
ros2 node info /my_node

# Topics
ros2 topic list -t                  # -t appends the message type
ros2 topic echo /scan               # print messages as they arrive
ros2 topic hz /scan                 # measure publish rate
ros2 topic info /scan --verbose     # publishers, subscribers, QoS profiles
ros2 topic pub /cmd_vel geometry_msgs/msg/Twist "{linear: {x: 0.2}}"

# Services and actions
ros2 service list -t
ros2 service call /reset std_srvs/srv/Empty
ros2 action list -t
ros2 action send_goal /navigate_to_pose nav2_msgs/action/NavigateToPose "{...}"

# Parameters
ros2 param list
ros2 param get /my_node use_sim_time
ros2 param set /my_node speed 0.5

# Message definitions
ros2 interface show geometry_msgs/msg/Twist

# Running things
ros2 run my_pkg my_node
ros2 launch my_pkg bringup.launch.py

# Recording
ros2 bag record -a                  # all topics
ros2 bag play rosbag2_2026_01_01/
```

`ros2 topic info --verbose` is worth singling out: it prints the QoS profile of
every endpoint, which is how the silent incompatibility above gets diagnosed.

---


## Discovery and Domains

There is no `roscore`. Nodes find each other by multicast DDS discovery, which
means **any ROS 2 node on the same subnet joins the same graph by default**. Two
people debugging robots on one office network will see each other's topics.

`ROS_DOMAIN_ID` partitions the network. Set it to the same integer (0-101 is the
safe range) on every machine that should talk, and a different one elsewhere:

```bash
export ROS_DOMAIN_ID=42
```

To stop discovery leaving the machine entirely:

```bash
export ROS_LOCALHOST_ONLY=1     # Humble
export ROS_AUTOMATIC_DISCOVERY_RANGE=LOCALHOST   # Jazzy and later
```

The DDS implementation itself is swappable (`rmw_fastrtps_cpp` is the default,
`rmw_cyclonedds_cpp` the common alternative) via `RMW_IMPLEMENTATION`. Every
node in a graph must use implementations that interoperate, so change it
everywhere or nowhere.

---
